# Paper 5, RB-2 - single-head axis map

**Naming.** W1-W7 = September runs behind the paper. RB = November rebuttal runs. RB-0 = the single-head run
(files `w8_*`); the canonical RB-0 is the **cuDNN 9.10.2 re-run of 4 Nov** (the first RB-0, 2 Nov, cuDNN 9.26, is
superseded). RB-1 / RB-1b = the drift checks. This is **RB-2**; it writes to `./rb2_output/`.

**What it answers.** Figure 4d, Table 14 and §4.3 say which axis each head rebuilds, but a "head" there is one head
index across blocks 6-11. RB-2 measures the row and column probe cost of **every single (block, head)** in blocks
6-11 under the three windows the paper uses:

| window | blocks 0-5 | reference (probed at blocks 7-11 and the output of block 11) | used in the paper for |
|---|---|---|---|
| `cref` | all rotations off | Cref, + probe noise | SSDC ranking (§4.3), top-1 (Table 18, RB-0) |
| `row` | only the row planes off | Dref_row, + probe noise | row cost (Figure 4d x-axis, Table 14) |
| `col` | only the column planes off | Dref_col, + probe noise | column cost (Figure 4d y-axis, Table 14) |

In each window, one (block, head) in 6-11 is also kept without rotations. Heads in blocks 6-10 are probed entering
block 11 and at the output of block 11; block-11 heads only at the output of block 11 (the input of block 11 cannot
see them). SSDC is recorded at all 13 readouts.

From this map: (1) which single heads are row or column specialists; (2) how much of each Table 14 index group
comes from one block; (3) whether the paper's two exceptions (DINOv3 head 1 costs both axes; the top head of
naver ViT-S costs neither) survive at single-head level; (4) whether a probe score predicts each head's top-1
cost (from RB-0) better than SSDC does, which fixes, by a rule written below before the run, how later runs pick
"top" heads.

**Coherence.** Same repack, 1,000 images, batch 256, seed 20260908, fp32, RPI, readout sites and probe protocol as
W5 and RB-0, in `env_paper5_cudnn910` (cuDNN 9.10.2), which reproduced September bit for bit in RB-1b. Gates demand
**bit-identical** agreement with W5 (reference SSDC and probes) and with the canonical RB-0 (every Cref-window
single head's SSDC, every row/column single head RB-0 already ran, and the probe noise). RB-2 adds probes to
conditions whose SSDC is already known, and needs nothing from W7.

**How to run.** Set `RB2_RB0_DIR` to the canonical RB-0 folder. Split with `RB2_MODELS`; each model saves after
every block, and re-running with `RB2_RESUME=1` (default) continues where it stopped.

In [1]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "datasets", "scipy", "pandas"]:
    ensure(p)
print("environment ready")

/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


environment ready


In [2]:
import os, json, time, warnings, traceback, shutil, hashlib, platform, socket
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
warnings.filterwarnings("ignore", category=UserWarning)
print("python", sys.executable)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f"memory: {free/1e9:.1f} / {total/1e9:.1f} GB free")

python /workspace/NRDL/env_paper5_cudnn910/bin/python
torch 2.10.0+cu128 | timm 1.0.29 | cuda True


device: NVIDIA H100 80GB HBM3


memory: 84.5 / 85.0 GB free


In [3]:
# config
REPO_PATH  = os.environ.get("RB2_REPO", "")        # .../vit-sae-analysis/project_code/src ("" = search)
RB0_DIR    = os.environ.get("RB2_RB0_DIR", "")     # the canonical RB-0 folder (cuDNN 9.10.2 re-run, 4 Nov)
MODELS_RUN = [m for m in os.environ.get("RB2_MODELS", "naver,DINOv3,small,reg1_gap").split(",") if m]
WINDOWS    = [w for w in os.environ.get("RB2_WINDOWS", "cref,row,col").split(",") if w]
RESUME     = os.environ.get("RB2_RESUME", "1") == "1"
STRICT     = os.environ.get("RB2_STRICT", "1") == "1"   # 1: a model whose reference gates fail is not continued
TEST_MODE  = os.environ.get("RB2_TEST", "0") == "1"     # CPU smoke test on tiny random models, not for results
OUT_DIR    = Path(os.environ.get("RB2_OUT", "rb2_output"))

N_SSDC, BATCH_SIZE = 1000, 256                      # W5 / RB-0, unchanged
METRIC, TORCH_SEED, INPUT_SIZE = "manhattan", 20260908, 224
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"  # every W2-W6 and RB-0 SSDC/probe number

N_BLOCKS = 12
WINDOW   = list(range(0, 6))      # rotations (or one axis's planes) off here, as in W3-W5 and RB-0
POST     = list(range(6, 12))     # where rebuilding happens
FINAL    = 12                     # readout key: output of block 11 after the model's final LayerNorm (as RB-0)

PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC = 5e-3, 20, 512, 0.8   # W2-W6 / RB-0 protocol, unchanged
PROBE_CHUNK, PROBE_DEVICE = 8192, None
NOISE_SEEDS, NOISE_BOOT = [0, 1, 2], 200                                       # W5 / RB-0 probe-noise recipe
P_REF = [7, 8, 9, 10, 11, FINAL]   # references: blocks 7-11 (the Table 14 average) and the output of block 11
P_SINGLE = [11, FINAL]             # single heads in blocks 6-10 (RB-0's readouts); block 11: [FINAL] only
NOISE_AT = (11, FINAL)             # probe noise on the three window references, as RB-0 part B

EXACT_TOL = 0.0                    # W5 and the canonical RB-0 ran in this environment: agreement must be bit for bit
EXPECT_TORCH, EXPECT_CUDNN = "2.10.0+cu128", 91002
EXPECT_CUDNN_PKGS = {"nvidia-cudnn-cu12": "9.10.2.21"}
# RB-1b (cudnn910_default_a): patch-embedding output vs float64 on the first 256-image batch
FIRST_LAYER_910 = {"naver": 0.00041303808169361526, "DINOv3": 0.0004745740108653513,
                   "small": 0.000510305810057617, "reg1_gap": 0.00026782853342761795}
# sha256 of the canonical RB-0 files (cuDNN 9.10.2 re-run, 4 Nov 2026)
RB0_SHA256 = {
    "w8_partA_naver.json":     "f50b694d24279ce63b8dac8da1906b7e60ab6522ce8de4c992b1c87ceed10914",
    "w8_partA_DINOv3.json":    "f1d963652ac3867c79bb170da2b2dd86ff8e978008f5ff59304a69947bb909b5",
    "w8_partA_small.json":     "be01ed6297ef819a8fc219853c07fc9a81a76d2c6ffbe9da8501379fdf6acbbb",
    "w8_partA_reg1_gap.json":  "da5ae49a2abc21aa0df8f4c43b249727a7edfe444254e239e637ed96a312fefb",
    "w8_partB_naver.json":     "410d21697751cd8eba338e593153f5b92acb0c421940a1580f96325e1d1dbd9a",
    "w8_partB_DINOv3.json":    "ffb5b84f0b7c5a6208d6b2f7a9132393b601076bb0f7748f9b66b9ba56a470bb",
    "w8_partB_small.json":     "411cc73401b18db8b9a884271dcb0768aae4e47a183326fdff1102f21e7f627a",
    "w8_partB_reg1_gap.json":  "a7a6481ff165c1c1d30e75458655511285b1262727db0d77905d6e7d69268ba5",
    "w8_partC_naver.json":     "c983414404ca645bc8144a2bd4ec76bfb583f4c5c9588ec8faffe853e5c3b985",
    "w8_partC_small.json":     "bf895274486dfaa9ab042118e15bb7c9863a5f6201ab27ec11fb539ecf7126f6",
    "w8_partC_reg1_gap.json":  "8b5641235e05b20bfdc597032097b2f85878c228bdc32ec8b63888227e393698",
    "w8_perimage_naver.npz":   "5fe4a53ecbf99bc583c56ef6ec9f774c5bd6027e1734dce16fe0c230c36358c6",
    "w8_perimage_small.npz":   "b9f6a3833766b73618c82b3e5ecbb81af672b11eaa49fab503e27bb8734cf40b",
    "w8_perimage_reg1_gap.npz": "10310f0c0109d65f0b594f7774006ddd2c69f5fc74438b13d32956d39738643e",
}

MODEL_SPECS = {
    "naver":    dict(name="vit_base_patch16_rope_224.naver_in1k",        classifier=True,  label="naver ViT-B"),
    "DINOv3":   dict(name="vit_base_patch16_dinov3.lvd1689m",            classifier=False, label="DINOv3 ViT-B"),
    "small":    dict(name="vit_small_patch16_rope_224.naver_in1k",       classifier=True,  label="naver ViT-S"),
    "reg1_gap": dict(name="vit_base_patch16_rope_reg1_gap_256.sbb_in1k", classifier=True,  label="reg1-gap ViT-B"),
}
RB0_FILES = {k: [f"w8_partA_{k}.json", f"w8_partB_{k}.json"] +
                ([f"w8_partC_{k}.json", f"w8_perimage_{k}.npz"] if s["classifier"] else [])
             for k, s in MODEL_SPECS.items()}
WINDOW_ORDER = ["cref", "row", "col"]
REFN = {"cref": "Cref", "row": "Dref_row", "col": "Dref_col"}

# rules, fixed before the run
RULE_BAR = ("noise bar = 3 x the larger combined probe SD (seed + image bootstrap, the W5 recipe) at that readout: "
            "row/col windows use Dref_row's row probe and Dref_col's column probe (RB-0's bar); the cref window uses "
            "Cref's row and column probes")
RULE_LABEL = ("a single head 'favours' rows if its row cost (row window, row probe) is above the bar AND exceeds its "
              "column cost (col window, column probe) by more than the bar; columns likewise; 'mixed' if a cost is "
              "above the bar but neither gap is; 'none' if both costs are within the bar (the grey of Figure 4d)")
RULE_READOUT = "primary readout: entering block 11 for heads in blocks 6-10; output of block 11 for block-11 heads"
RULE_SCORES = {
    "ssdc_b11":  "SSDC lost entering block 11 vs Cref (the paper's rebuilding score applied to one head)",
    "probe_b11": "row + column probe points lost entering block 11 vs Cref, in the cref window (primary probe score)",
    "axis_b11":  "row cost (row window) + column cost (col window) entering block 11 (Figure 4d's two axes, summed)",
}
RULE_NEXT = ("Later runs that pick 'top' single heads (RB-3 onward) rank heads in blocks 6-10 by probe_b11 if, on at "
             "least two of naver ViT-B, naver ViT-S and reg1-gap, the 95% bootstrap interval (2,000 resamples of the "
             "heads) of rho(probe_b11, top-1 cost) - rho(ssdc_b11, top-1 cost) lies above zero; otherwise by ssdc_b11. "
             "Block-11 heads are ranked separately, at the output of block 11. Top-1 cost = RB-0 part C.")
DELTA_BOOT, DELTA_SEED, PERM_N = 2000, 2026, 10000

if TEST_MODE:
    N_SSDC, BATCH_SIZE = 24, 8
    PROBE_PASSES, NOISE_BOOT, DELTA_BOOT, PERM_N = 2, 10, 200, 500
    OUT_DIR = Path(os.environ.get("RB2_OUT", "rb2_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)
_bad = [m for m in MODELS_RUN if m not in MODEL_SPECS] + [w for w in WINDOWS if w not in WINDOW_ORDER]
if _bad:
    raise SystemExit(f"Unknown entries in RB2_MODELS / RB2_WINDOWS: {_bad}")
WINDOWS = [w for w in WINDOW_ORDER if w in WINDOWS]

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDEV = PROBE_DEVICE or DEVICE
GATES = []
GATE_FILE = OUT_DIR / f"rb2_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))   # kept per session, so a resumed run still reports every gate
    return bool(passed) or skip
print(f"device {DEVICE} | probes on {PDEV} | fp32 | models {MODELS_RUN} | windows {WINDOWS}")
print(f"SSDC/probes: {SSDC_DATASET} x {N_SSDC} | batch {BATCH_SIZE} | seed {TORCH_SEED}")
print(f"output {OUT_DIR.resolve()} | resume {RESUME} | strict {STRICT} | test mode {TEST_MODE} | run {RUN_STAMP}")

device cuda | probes on cuda | fp32 | models ['naver', 'DINOv3', 'small', 'reg1_gap'] | windows ['cref', 'row', 'col']
SSDC/probes: benjamin-paine/imagenet-1k-256x256 x 1000 | batch 256 | seed 20260908
output /workspace/NRDL/experiments/rb2/rb2_output | resume True | strict True | test mode False | run 2026-11-05T04-31-07Z


## Pre-flight: are we in `env_paper5_cudnn910`?
A small GPU convolution loads cuDNN; then the notebook checks the version PyTorch reports, which library files were
actually loaded (from the process's own memory map), the installed cuDNN packages, and that the TF32 / cuDNN flags
are PyTorch's defaults (W5 and RB-0 ran with defaults). Nothing is measured if any check fails.

In [4]:
def _sh(cmd, timeout=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return ((r.stdout or "") + (r.stderr or "")).strip()
    except Exception as ex:
        return f"(failed: {ex!r})"

def _loaded_cudnn():
    try:
        with open("/proc/self/maps") as f:
            return sorted({l.split()[-1] for l in f if "libcudnn" in l and l.split()[-1].startswith("/")})
    except Exception:
        return []

import importlib.metadata as _md
if DEVICE == "cuda":
    _x = torch.randn(2, 3, 32, 32, device="cuda"); _w = torch.randn(8, 3, 16, 16, device="cuda")
    F.conv2d(_x, _w, stride=16); torch.cuda.synchronize(); del _x, _w
CUDNN_PKGS = {}
for _d in _md.distributions():
    _n = (_d.metadata.get("Name") or "").lower()
    if "cudnn" in _n: CUDNN_PKGS[_n] = _d.version
ENV = {
    "python": sys.executable, "prefix": sys.prefix, "host": socket.gethostname(), "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__,
    "cudnn_reported": torch.backends.cudnn.version() if DEVICE == "cuda" else None,
    "libcudnn_loaded": _loaded_cudnn(), "cudnn_packages": CUDNN_PKGS,
    "flags": {"matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32, "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
              "cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic,
              "float32_matmul_precision": torch.get_float32_matmul_precision()},
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "gpu": _sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,memory.used,memory.total --format=csv") if DEVICE == "cuda" else "cpu",
}
(OUT_DIR / f"rb2_pip_freeze_{RUN_STAMP}.txt").write_text(_sh(f"{sys.executable} -m pip freeze", timeout=300))
(OUT_DIR / f"rb2_environment_{RUN_STAMP}.json").write_text(json.dumps(ENV, indent=1, default=str))
print(json.dumps({k: v for k, v in ENV.items() if k != "gpu"}, indent=1, default=str)); print(ENV["gpu"])

_pre = []
if TEST_MODE:
    gate("pre-flight (cuDNN 9.10.2 environment)", "env", True, "test mode on CPU", skip=True)
else:
    _pre.append(gate("CUDA available", "env", DEVICE == "cuda"))
    _pre.append(gate(f"torch is {EXPECT_TORCH}", "env", torch.__version__ == EXPECT_TORCH, torch.__version__))
    _pre.append(gate(f"PyTorch reports cuDNN {EXPECT_CUDNN} (9.10.2)", "env", ENV["cudnn_reported"] == EXPECT_CUDNN,
                     str(ENV["cudnn_reported"])))
    _prefix = os.path.realpath(sys.prefix)
    _libs = ENV["libcudnn_loaded"]
    _pre.append(gate("every loaded libcudnn file comes from this environment's nvidia/cudnn folder", "env",
                     bool(_libs) and all(os.path.realpath(l).startswith(_prefix) and "/nvidia/cudnn/" in l for l in _libs),
                     "; ".join(Path(l).name for l in _libs) + f" (prefix {_prefix})"))
    _pre.append(gate("the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13)", "env",
                     CUDNN_PKGS == EXPECT_CUDNN_PKGS, str(CUDNN_PKGS)))
    _f = ENV["flags"]
    _pre.append(gate("TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0)", "env",
                     _f["matmul_allow_tf32"] is False and _f["cudnn_allow_tf32"] is True and _f["cudnn_benchmark"] is False
                     and _f["cudnn_deterministic"] is False and _f["float32_matmul_precision"] == "highest", str(_f)))
PREFLIGHT_OK = all(_pre)
if not PREFLIGHT_OK:
    raise SystemExit("PRE-FLIGHT FAILED: this notebook must run in env_paper5_cudnn910 (cuDNN 9.10.2), with PyTorch's "
                     "default flags. Nothing was measured. Switch the kernel / python and run again.")
print("\npre-flight OK")

{
 "python": "/workspace/NRDL/env_paper5_cudnn910/bin/python",
 "prefix": "/workspace/NRDL/env_paper5_cudnn910",
 "host": "gpu-pod",
 "platform": "Linux-6.8.0-41-generic-x86_64-with-glibc2.35",
 "torch": "2.10.0+cu128",
 "timm": "1.0.29",
 "numpy": "2.4.6",
 "cudnn_reported": 91002,
 "libcudnn_loaded": [
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_engines_precompiled.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_engines_runtime_compiled.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_graph.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_heuristic.so.9"
 ],
 "cudnn_packages": {
  "nvidia-cudnn-cu12": "9.10.2.21"
 },
 "flags": {
  "matmul_allow_tf32": false,
  "cudnn_allow_tf32": t

  [PASS] env: torch is 2.10.0+cu128: 2.10.0+cu128


  [PASS] env: PyTorch reports cuDNN 91002 (9.10.2): 91002


  [PASS] env: every loaded libcudnn file comes from this environment's nvidia/cudnn folder: libcudnn.so.9; libcudnn_engines_precompiled.so.9; libcudnn_engines_runtime_compiled.so.9; libcudnn_graph.so.9; libcudnn_heuristic.so.9 (prefix /workspace/NRDL/env_paper5_cudnn910)


  [PASS] env: the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13): {'nvidia-cudnn-cu12': '9.10.2.21'}


  [PASS] env: TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0): {'matmul_allow_tf32': False, 'cudnn_allow_tf32': True, 'cudnn_benchmark': False, 'cudnn_deterministic': False, 'float32_matmul_precision': 'highest'}



pre-flight OK


In [5]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB2_REPO. Tried: " +
                            ", ".join(str(c) for c in cands[:8]))

repo = resolve_repo()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
REPO_COMMIT = _sh(f"git -C {repo} rev-parse HEAD")
print("repo root:", repo, "| commit", REPO_COMMIT)

from metrics.ssdc import (evaluate_ssdc, spatial_similarity_distance_correlation, SSDCAccumulator,
                          _register_ssdc_accumulator_hooks)
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope
from main.model import predict
from main.prep_data import prep_data
from experiments.common import load_imagenet

from scipy.spatial.distance import cdist as _cdist
from scipy.stats import rankdata
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
_r = spatial_similarity_distance_correlation(np.random.default_rng(0).normal(size=(_G * _G,) * 2),
                                             grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric",
     _a > 0.999 and _b < -0.999 and abs(_r) < 0.1, f"{_a:+.3f}/{_b:+.3f}/{_r:+.3f}")

repo root: /workspace/NRDL/clone/vit-sae-analysis/project_code/src | commit 28e0856cd87c8582aff2783368b3d2f098ae3190


  [PASS] repo-metric: imported metric satisfies its unit test: +1.000/-1.000/-0.005


True

## The canonical RB-0 (fingerprint check)
RB-2 is gated against RB-0 and its analysis uses RB-0's top-1 per head, so it must read the **cuDNN 9.10.2 re-run**
(`rb0_cudnn910/w8_output`). The sha256 of every file it reads is checked against the fingerprints above, so the
superseded first RB-0 (cuDNN 9.26) cannot be used by mistake.

In [6]:
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

RB0 = {}
if not RB0_DIR and not TEST_MODE:
    raise SystemExit("Set RB2_RB0_DIR to the canonical RB-0 folder: "
                     "/workspace/NRDL/experiments/rb0_cudnn910/w8_output")
_rb0 = Path(RB0_DIR).expanduser() if RB0_DIR else None
_rb0_bad = []
for key in MODELS_RUN:
    for fn in RB0_FILES[key]:
        p = (_rb0 / fn) if _rb0 else None
        if p is None or not p.exists():
            if TEST_MODE:
                gate(f"RB-0 file {fn} present", key, True, "test mode: no RB-0 folder", skip=True)
            else:
                gate(f"RB-0 file {fn} present", key, False, str(p)); _rb0_bad.append(fn)
            continue
        if TEST_MODE:
            gate(f"{fn} is the canonical RB-0 file", key, True, "test mode: hash not checked", skip=True)
        else:
            d = sha256_file(p)
            if not gate(f"{fn} is the canonical RB-0 file (cuDNN 9.10.2 re-run)", key, d == RB0_SHA256[fn],
                        f"sha256 {d[:16]}... (expected {RB0_SHA256[fn][:16]}...)"):
                _rb0_bad.append(fn)
if _rb0_bad:
    raise SystemExit(f"RB-0 FINGERPRINT FAILED for {_rb0_bad}. RB2_RB0_DIR must be the cuDNN 9.10.2 re-run "
                     "(rb0_cudnn910/w8_output, 4 Nov). The first RB-0 (2 Nov, cuDNN 9.26) is superseded. Nothing was measured.")
for key in MODELS_RUN:
    d = {}
    for part in "ABC":
        p = (_rb0 / f"w8_part{part}_{key}.json") if _rb0 else None
        d[part] = json.loads(p.read_text()) if (p is not None and p.exists()) else None
    p = (_rb0 / f"w8_perimage_{key}.npz") if _rb0 else None
    if p is not None and p.exists():
        z = np.load(p); d["P"] = {n: z[n] for n in z.files}
    else:
        d["P"] = None
    RB0[key] = d
    if d["A"] is not None:
        gate("RB-0 parts A/B (and C) are complete", key,
             all(d[x] is None or d[x].get("complete") for x in "ABC") and d["B"] is not None,
             ", ".join(f"{x}={'complete' if (d[x] or {}).get('complete') else 'missing'}" for x in "ABC"))
print("RB-0 loaded:", {k: [x for x in "ABCP" if v[x] is not None] for k, v in RB0.items()})

  [PASS] naver: w8_partA_naver.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 f50b694d24279ce6... (expected f50b694d24279ce6...)


  [PASS] naver: w8_partB_naver.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 410d21697751cd8e... (expected 410d21697751cd8e...)


  [PASS] naver: w8_partC_naver.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 c983414404ca645b... (expected c983414404ca645b...)


  [PASS] naver: w8_perimage_naver.npz is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 5fe4a53ecbf99bc5... (expected 5fe4a53ecbf99bc5...)


  [PASS] DINOv3: w8_partA_DINOv3.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 f1d963652ac3867c... (expected f1d963652ac3867c...)


  [PASS] DINOv3: w8_partB_DINOv3.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 ffb5b84f0b7c5a62... (expected ffb5b84f0b7c5a62...)


... (9 outputs left out)

  [PASS] DINOv3: RB-0 parts A/B (and C) are complete: A=complete, B=complete, C=missing


  [PASS] small: RB-0 parts A/B (and C) are complete: A=complete, B=complete, C=complete


  [PASS] reg1_gap: RB-0 parts A/B (and C) are complete: A=complete, B=complete, C=complete


RB-0 loaded: {'naver': ['A', 'B', 'C', 'P'], 'DINOv3': ['A', 'B'], 'small': ['A', 'B', 'C', 'P'], 'reg1_gap': ['A', 'B', 'C', 'P']}


In [7]:
# committed W5 anchors (read from the repo's result files, never retyped)
RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"
def _first(pattern):
    hits = sorted(RES_ROOT.glob(pattern)); return hits[0] if hits else None
W5_FILES = {nm: _first(f"w5/w5_partA_{nm}_*.json") for nm in MODEL_SPECS}
HEADLINE = {"naver": (0.329663, 0.469422), "DINOv3": (0.442640, 0.948654), "small": (0.244217, 0.371434),
            "reg1_gap": (0.244575, 0.689023)}   # (Cref b11, baseline b4), typed only to prove the right files were found
W5_REF = {"baseline": "baseline", "Cref": "Cref_prefix5", "Dref_row": "Dref_axisrow5", "Dref_col": "Dref_axiscol5"}
ANCH = {}
if TEST_MODE:
    gate("committed W5 anchors loaded", "W5", True, "test mode: tiny random models have no W5 anchors", skip=True)
for nm in ([] if TEST_MODE else MODELS_RUN):
    f = W5_FILES.get(nm)
    if f is None:
        gate("committed W5 file found", nm, False, "missing"); continue
    e = json.load(open(f)); c = e["conditions"]
    a = {"file": f.name, "n_heads": e["n_heads"], "row_planes": e["row_planes"], "col_planes": e["col_planes"],
         "stageC": [d["head"] for d in e["stageC_ranking"]], "ssdc": {}, "probes": {}, "noise": {}, "D": {}}
    for ours, theirs in W5_REF.items():
        a["ssdc"][ours] = c[theirs]["ssdc"]
        a["probes"][ours] = {int(b): {ax: v[ax] for ax in ("row", "col")} for b, v in c[theirs]["probes"].items() if 7 <= int(b) <= 11}
        a["noise"][ours] = {int(b): v["noise"] for b, v in c[theirs]["probes"].items() if "noise" in v and int(b) == 11}
    for k in c:
        if k.startswith("D_row_head") or k.startswith("D_col_head"):
            a["D"].setdefault(int(k[-2:]), {})[k[2:5]] = {int(b): {ax: v[ax] for ax in ("row", "col")} for b, v in c[k]["probes"].items()}
    ANCH[nm] = a
    cb11, b4 = HEADLINE[nm]
    gate("committed W5 file matches its headline values", nm,
         abs(a["ssdc"]["Cref"][11] - cb11) < 1e-6 and abs(a["ssdc"]["baseline"][4] - b4) < 1e-6, f.name)
print("W5 anchors:", {k: v["file"] for k, v in ANCH.items()})
print("Table 14 index groups in W5:", {k: sorted(v["D"]) for k, v in ANCH.items()})

  [PASS] naver: committed W5 file matches its headline values: w5_partA_naver_2026-09-13T10-16-41Z.json


  [PASS] DINOv3: committed W5 file matches its headline values: w5_partA_DINOv3_2026-09-13T10-16-41Z.json


  [PASS] small: committed W5 file matches its headline values: w5_partA_small_2026-09-13T10-16-41Z.json


  [PASS] reg1_gap: committed W5 file matches its headline values: w5_partA_reg1_gap_2026-09-13T10-16-41Z.json


W5 anchors: {'naver': 'w5_partA_naver_2026-09-13T10-16-41Z.json', 'DINOv3': 'w5_partA_DINOv3_2026-09-13T10-16-41Z.json', 'small': 'w5_partA_small_2026-09-13T10-16-41Z.json', 'reg1_gap': 'w5_partA_reg1_gap_2026-09-13T10-16-41Z.json'}
Table 14 index groups in W5: {'naver': [1, 2, 6, 11], 'DINOv3': [1, 2, 9, 10], 'small': [0, 2, 4], 'reg1_gap': [3, 6, 7, 9]}


In [8]:
# dataset: explicit id, no silent fallback
if TEST_MODE:
    from PIL import Image as _PIL
    class _FakeStream:
        def __init__(self, n, seed):
            self.n, self.seed = n, seed
        def __iter__(self):
            rng = np.random.default_rng(self.seed)
            for i in range(self.n):
                arr = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
                yield {"image": _PIL.fromarray(arr), "label": int(i % 10)}
    DS_SSDC = _FakeStream(400, 1)
    print("TEST MODE: random images stand in for the repack")
else:
    DS_SSDC = load_imagenet(split="validation", streaming=True, dataset_id=SSDC_DATASET, fallback_id=None)
    gate("SSDC dataset is the repack every W2-W6 and RB-0 SSDC/probe number used", "data",
         SSDC_DATASET == "benjamin-paine/imagenet-1k-256x256", SSDC_DATASET)

loaded benjamin-paine/imagenet-1k-256x256 [validation]
  [PASS] data: SSDC dataset is the repack every W2-W6 and RB-0 SSDC/probe number used: benjamin-paine/imagenet-1k-256x256


In [9]:
# rotary helpers: from W5 / RB-0
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None

def attn_attrs(model, source):
    rh, src, nh = False, "default(interleaved)", None
    for blk in get_vit_blocks(model, source):
        a = get_block_attention(blk, source)
        for at in ("rotate_half", "half", "rope_rotate_half"):
            v = getattr(a, at, None)
            if isinstance(v, bool): rh, src = v, at; break
        nh = getattr(a, "num_heads", None); break
    return rh, src, nh

def plane_columns(p, n, rh): return [p, p + n] if rh else [2 * p, 2 * p + 1]

def derive_axis(emb, n_planes, rh, H, W):
    sin, cos = emb.detach().float().chunk(2, -1); rows = []; dec = True
    for p in range(n_planes):
        c = plane_columns(p, n_planes, rh)[0]
        z = torch.complex(cos[:, c], sin[:, c]).reshape(H, W)
        ra = (z - z.mean(dim=1, keepdim=True)).abs().mean().item()
        rb = (z - z.mean(dim=0, keepdim=True)).abs().mean().item()
        ax = "row" if ra < rb else "col"; lo, hi = sorted([ra, rb]); r = lo / hi if hi > 1e-12 else 1.0
        if r > 0.10: dec = False
        rows.append({"plane": p, "axis": ax, "ratio": r})
    return rows, [r["plane"] for r in rows if r["axis"] == "row"], [r["plane"] for r in rows if r["axis"] == "col"], dec

def mask_planes(emb, kill, n, rh):
    if not kill: return emb
    s, c = emb.chunk(2, -1); s, c = s.clone(), c.clone()
    for p in kill:
        for col in plane_columns(p, n, rh): s[..., col] = 0.0; c[..., col] = 1.0
    return torch.cat([s, c], -1)

def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
IDEN = identity_like

def per_head(heads, inner, n_heads):
    hs = sorted(set(heads))
    def tf(emb):
        if emb.dim() != 2:
            raise RuntimeError(f"per_head expected 2-D rope, got {tuple(emb.shape)}: stacked overrides")
        out = emb.unsqueeze(0).repeat(n_heads, 1, 1)
        if hs:
            t = inner(emb)
            for h in hs: out[h] = t
        return out
    return tf

class RopeComposite:
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source
        s.segs = [(list(b), t) for b, t in segs]; seen = set()
        for b, _ in s.segs:
            if seen & set(b): raise ValueError(f"overlapping segments: {sorted(seen & set(b))}")
            seen |= set(b)
        s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source)
                                 .register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

def win_plus(model, source, heads_by_block, n_heads, window_tf=IDEN, inner=IDEN, window=None):
    """Rotations off in the window (window_tf), plus `inner` on the listed heads. heads_by_block: {block: [heads]},
    blocks outside the window."""
    segs = [(list(WINDOW if window is None else window), window_tf)] if (window is None or window) else []
    for k in sorted(heads_by_block):
        hs = sorted(set(heads_by_block[k]))
        if hs: segs.append(([k], per_head(hs, inner, n_heads)))
    return RopeComposite(model, source, segs)

In [10]:
# algebra gates (as in W5 / RB-0)
def _rot_il(x):
    x = x.unflatten(-1, (-1, 2)); return torch.stack([-x[..., 1], x[..., 0]], -1).flatten(-2)
def _rot_h(x):
    a, b = x.chunk(2, -1); return torch.cat([-b, a], -1)
def _apply(x, e, half=False):
    s, c = e.chunk(2, -1); return x * c + (_rot_h(x) if half else _rot_il(x)) * s

for lname, rh in [("interleaved", False), ("rotate_half", True)]:
    npl, dim, H_ = 32, 64, 12
    e = torch.randn(196, 2 * dim); x = torch.randn(2, H_, 196, dim)
    gate(f"empty plane mask bit-identical [{lname}]", "algebra", torch.equal(mask_planes(e, [], npl, rh), e))
    gate(f"full plane mask equals identity [{lname}]", "algebra",
         torch.allclose(mask_planes(e, list(range(npl)), npl, rh), identity_like(e)))
    m = mask_planes(e, [3], npl, rh)
    touched = torch.nonzero((m - e).abs().sum(0) > 0).flatten().tolist()
    sc = plane_columns(3, npl, rh); exp = sorted(set(sc + [c + dim for c in sc]))
    gate(f"plane 3 touches only its own columns [{lname}]", "algebra", touched == exp, f"{touched} vs {exp}")
    full = _apply(x, e, rh)
    gate(f"mask NO heads equals unmodified rope [{lname}]", "per-head",
         torch.allclose(_apply(x, per_head([], identity_like, H_)(e), rh), full, atol=1e-6))
    one = _apply(x, per_head([3], identity_like, H_)(e), rh); others = [h for h in range(H_) if h != 3]
    gate(f"mask head 3 leaves others untouched [{lname}]", "per-head", torch.allclose(one[:, others], full[:, others], atol=1e-6))
    gate(f"mask head 3 changes head 3 [{lname}]", "per-head", not torch.allclose(one[:, 3], full[:, 3]))
    # single head off after a one-axis window (the window masks that axis's planes, the head masks all)
    ew = mask_planes(e, list(range(0, npl, 2)), npl, rh)
    gate(f"one-axis window leaves the other axis's planes bit-identical [{lname}]", "algebra",
         torch.equal(ew.chunk(2, -1)[0][:, plane_columns(1, npl, rh)], e.chunk(2, -1)[0][:, plane_columns(1, npl, rh)]))

class _Stub:
    def __init__(s, n=12): s.blocks = [nn.Identity() for _ in range(n)]
try:
    _wp = win_plus(_Stub(), "timm", {7: [3]}, 12, window_tf=IDEN)
    dj = [b for b, _ in _wp.segs] == [WINDOW, [7]]
except Exception:
    dj = False
gate("win_plus builds the window plus one segment for the single head's block", "composite", dj)
try:
    win_plus(_Stub(), "timm", {5: [0]}, 12); bad = False
except ValueError:
    bad = True
gate("win_plus REJECTS a single head inside the window", "composite", bad)

  [PASS] algebra: empty plane mask bit-identical [interleaved]


  [PASS] algebra: full plane mask equals identity [interleaved]


  [PASS] algebra: plane 3 touches only its own columns [interleaved]: [6, 7, 70, 71] vs [6, 7, 70, 71]


  [PASS] per-head: mask NO heads equals unmodified rope [interleaved]


  [PASS] per-head: mask head 3 leaves others untouched [interleaved]


  [PASS] per-head: mask head 3 changes head 3 [interleaved]


... (7 outputs left out)

  [PASS] algebra: one-axis window leaves the other axis's planes bit-identical [rotate_half]


  [PASS] composite: win_plus builds the window plus one segment for the single head's block


  [PASS] composite: win_plus REJECTS a single head inside the window


True

In [11]:
# image cache: decode once per model, replay through a real DataLoader (RB-0)
class _CachedBatches(torch.utils.data.IterableDataset):
    def __init__(self, batches): super().__init__(); self.batches = batches
    def __iter__(self):
        for b in self.batches: yield b

def cached_loader(cache):
    return torch.utils.data.DataLoader(_CachedBatches(cache), batch_size=None, num_workers=0,
                                       pin_memory=torch.cuda.is_available(), shuffle=False)

def build_cache(dataset, processor, source, n):
    t = time.time()
    dl = prep_data(dataset, processor, source, corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(im.clone(), torch.as_tensor(lb).clone()) for im, lb in dl]
    got = sum(int(lb.numel()) for _, lb in out)
    print(f"    cached {got} images in {time.time() - t:.0f}s")
    return out

def cache_digest(cache):
    h = hashlib.sha1()
    for im, lb in cache[:2]:
        h.update(im[:4].numpy().tobytes()); h.update(lb[:4].numpy().tobytes())
    return h.hexdigest()[:12]

In [12]:
# measurement: RB-0's measure() without the bootstrap / head-norm extras
def final_norm_module(model):
    n = getattr(model, "norm", None)
    if n is not None and not isinstance(n, nn.Identity): return n, "norm"
    f = getattr(model, "fc_norm", None)
    if f is not None and not isinstance(f, nn.Identity): return f, "fc_norm"
    return nn.Identity(), "identity"

def measure(M, ctx, capture=()):
    """One RPI sweep over the cached images under `ctx`: SSDC at every block input and at FINAL, plus captured patch
    tokens."""
    model, source, n_prefix = M["model"], M["source"], M["n_prefix"]
    cache = M["cache"]["ssdc"]
    blocks = get_vit_blocks(model, source)
    acc = SSDCAccumulator(n_prefix=n_prefix, spatial_metric=METRIC)
    handles = _register_ssdc_accumulator_hooks(model, source, acc)        # the repo's own readout
    fnorm = M["final_norm"]
    st = {"next": 0, "cur": 0}
    cap = {r: [] for r in capture}
    def keep(r, tok):
        if r in cap: cap[r].append(tok[:, n_prefix:, :].detach().float().cpu())
    extra = [model.register_forward_pre_hook(
        lambda m, args: st.update(cur=st["next"], next=st["next"] + int(args[0].shape[0])))]
    for i in range(N_BLOCKS):
        if i in cap:
            extra.append(get_block_attention(blocks[i], source).register_forward_hook(
                lambda m, inp, out, i=i: keep(i, inp[0].detach())))
    def final_hook(m, inp, out):
        t = fnorm(out.detach()); acc.add(FINAL, t); keep(FINAL, t)
    extra.append(blocks[-1].register_forward_hook(final_hook))
    t0 = time.time()
    torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            predict(model, cached_loader(cache), source, RPI=True, half=False)
    finally:
        for h in handles + extra: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(N_BLOCKS)] + [acc.ssdc(FINAL, remove_prefix=True)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": int(st["next"]),
           "runtime_s": round(time.time() - t0, 1)}
    if capture:
        res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    return res

# probes: W5's per-block-seeded protocol from RB-0
def _fit(Xtr, ytr, Xte, yte, nc, seed):
    torch.manual_seed(seed)
    head = nn.Linear(Xtr.shape[1], nc).to(Xtr.device)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR)
    n = Xtr.shape[0]; peak = 0.0; best = None
    for _ in range(PROBE_PASSES):
        perm = torch.randperm(n, device=Xtr.device); head.train()
        for s0 in range(0, n, PROBE_BATCH):
            b = perm[s0:s0 + PROBE_BATCH]
            opt.zero_grad(); F.cross_entropy(head(Xtr[b]), ytr[b]).backward(); opt.step()
        head.eval(); correct = 0; preds = []
        with torch.no_grad():
            for s0 in range(0, Xte.shape[0], PROBE_CHUNK):
                p = head(Xte[s0:s0 + PROBE_CHUNK]).argmax(-1); preds.append(p)
                correct += (p == yte[s0:s0 + PROBE_CHUNK]).sum().item()
        acc_ = correct / Xte.shape[0]
        if acc_ > peak: peak = acc_; best = torch.cat(preds)
    del head, opt
    return float(peak), best

def _split(n_images, T, seed=TORCH_SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(n_images, generator=g); n_tr = int(round(n_images * PROBE_TRAIN_FRAC))
    rows = lambda ids: (ids.unsqueeze(1) * T + torch.arange(T).unsqueeze(0)).reshape(-1)
    return rows(idx[:n_tr]), rows(idx[n_tr:]), idx[n_tr:]

def probe_block(acts, H, W, block, heads=("exact", "row", "col"), noise=False):
    n_images, T, D = acts.shape
    feats = acts.reshape(-1, D)
    tr, te, te_img = _split(n_images, T)
    Xtr = feats[tr].to(PDEV); Xte = feats[te].to(PDEV)
    mu = Xtr.mean(0, keepdim=True); sd = Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    slot = torch.arange(T); out = {}
    labels = {"exact": (slot, T), "row": (slot // W, H), "col": (slot % W, W)}
    for name in heads:
        lab, nc = labels[name]; y = lab.repeat(n_images)
        acc_, preds = _fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block)
        out[name] = acc_
        if noise:
            corr = (preds == y[te].to(PDEV)).float()
            g = np.random.default_rng(TORCH_SEED + block)
            n_te = len(te_img); per = corr.reshape(n_te, T).mean(1).cpu().numpy()
            bt = [per[g.integers(0, n_te, n_te)].mean() for _ in range(NOISE_BOOT)]
            seeds = [_fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block + 1000 * s)[0]
                     for s in NOISE_SEEDS]
            out.setdefault("noise", {})[name] = {
                "boot_sd": float(np.std(bt, ddof=1)), "seed_sd": float(np.std(seeds, ddof=1)),
                "seed_values": [float(v) for v in seeds],
                "combined_sd": float(np.sqrt(np.var(bt, ddof=1) + np.var(seeds, ddof=1)))}
    out["chance"] = {"exact": 1.0 / T, "row": 1.0 / H, "col": 1.0 / W}
    del Xtr, Xte
    if PDEV == "cuda": torch.cuda.empty_cache()
    return out

In [13]:
# per-model setup
TEST_INIT_SEED = {"naver": 101, "DINOv3": 102, "small": 103, "reg1_gap": 104}
def load_model(key):
    spec = MODEL_SPECS[key]
    if TEST_MODE:
        tiny = dict(embed_dim=96, num_heads=6 if key != "naver" else 12, pretrained=False, dynamic_img_size=True)
        if key == "naver": tiny["embed_dim"] = 192
        tiny["num_classes"] = 10 if spec["classifier"] else 0
        torch.manual_seed(TEST_INIT_SEED[key])      # same tiny weights in every process (test mode only)
        model = timm.create_model(spec["name"].split(".")[0], **tiny).to(DEVICE).eval()
        with torch.no_grad():   # random init uses layer-scale 1e-5 (a no-op network): open the branches, sharpen attention
            for blk in model.blocks:
                for gname in ("gamma_1", "gamma_2"):
                    g_ = getattr(blk, gname, None)
                    if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                q_ = getattr(blk.attn, "qkv", None)
                if q_ is not None: q_.weight.mul_(4.0)
                for pn in ("q_proj", "k_proj"):
                    m_ = getattr(blk.attn, pn, None)
                    if m_ is not None: m_.weight.mul_(4.0)
        cfg = timm.data.resolve_model_data_config(model); cfg["input_size"] = (3, INPUT_SIZE, INPUT_SIZE)
        return model, timm.data.create_transform(**cfg, is_training=False), "timm"
    return load_rope(model_name=spec["name"], device=DEVICE, half=False, input_size=INPUT_SIZE)

def first_layer_rel_diff(M):
    """RB-1b's check: patch-embedding output on the first 256-image batch vs a float64 computation."""
    model, source = M["model"], M["source"]
    conv = get_patch_embed_conv(model, source); grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    try:
        with torch.no_grad(): model(M["cache"]["ssdc"][0][0].to(DEVICE))
    finally:
        h.remove()
    with torch.no_grad():
        y64 = F.conv2d(grab["x"].double(), conv.weight.double(), None if conv.bias is None else conv.bias.double(),
                       conv.stride, conv.padding, conv.dilation, conv.groups)
        v = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
    n = int(grab["x"].shape[0]); del grab, y64
    return v, n

def setup_model(key):
    spec = MODEL_SPECS[key]
    print("\n" + "=" * 78 + f"\n{key}  ({spec['name']})\n" + "=" * 78, flush=True)
    model, processor, source = load_model(key)
    model.eval().float()
    rh, rh_src, nh_attr = attn_attrs(model, source)
    n_prefix = int(getattr(model, "num_prefix_tokens", 1))
    M = {"key": key, "model": model, "source": source, "processor": processor, "n_prefix": n_prefix,
         "rotate_half": bool(rh), "rotate_half_source": rh_src, "n_heads": int(nh_attr), "cache": {}}
    M["cache"]["ssdc"] = build_cache(DS_SSDC, processor, source, N_SSDC)
    with torch.no_grad():
        g = get_patch_embed_conv(model, source)(M["cache"]["ssdc"][0][0][:1].to(DEVICE))
    Hg, Wg = int(g.shape[-2]), int(g.shape[-1]); M["grid"] = (Hg, Wg); T = Hg * Wg
    gate("forward grid is 14x14", key, (Hg, Wg) == (14, 14), f"{Hg}x{Wg}")
    fn, fn_name = final_norm_module(model); M["final_norm"] = fn; M["final_norm_name"] = fn_name
    print(f"  heads={M['n_heads']} prefix={n_prefix} rotate_half={rh} final readout via model.{fn_name}")
    A_ = ANCH.get(key)
    if A_:
        gate("num_heads matches W5", key, M["n_heads"] == A_["n_heads"], f"{M['n_heads']} vs {A_['n_heads']}")
    ra = (RB0.get(key) or {}).get("A")
    if ra:
        gate("num_heads matches RB-0", key, M["n_heads"] == ra["n_heads"], f"{M['n_heads']} vs {ra['n_heads']}")

    rname, rmod = find_rope(model); M["rope_name"], M["rope"] = rname, rmod
    emb0 = rmod.get_embed(shape=(Hg, Wg))
    n_planes = int(emb0.shape[-1] // 4); M["n_planes"] = n_planes
    amap, rowp, colp, dec = derive_axis(emb0, n_planes, rh, Hg, Wg)
    M["row_planes"], M["col_planes"] = rowp, colp
    gate("rope rows equal patch tokens", key, int(emb0.shape[0]) == T, f"{emb0.shape[0]} vs {T}")
    gate("rotary table has no head dimension (every head in a block uses the same frequencies)", key,
         emb0.dim() == 2, f"shape {tuple(emb0.shape)}; {n_planes} planes per head")
    gate("every plane classifies decisively", key, dec, f"max ratio {max(a['ratio'] for a in amap):.1e}")
    gate("axis split is balanced", key, len(rowp) == len(colp) == n_planes // 2, f"{len(rowp)}/{len(colp)}")
    if A_:
        gate("row/col planes equal W5's runtime derivation", key,
             rowp == A_["row_planes"] and colp == A_["col_planes"], f"row {rowp[:3]}..")
    M["ROWM"] = lambda e, _r=rowp, _n=n_planes, _h=rh: mask_planes(e, _r, _n, _h)
    M["COLM"] = lambda e, _c=colp, _n=n_planes, _h=rh: mask_planes(e, _c, _n, _h)

    # readout-location gate: FINAL = final norm of block-11 output (RB-0)
    if fn_name == "norm":
        got = {}
        h1 = get_vit_blocks(model, source)[-1].register_forward_hook(lambda m, i, o: got.update(a=fn(o.detach())))
        h2 = model.norm.register_forward_hook(lambda m, i, o: got.update(b=o.detach()))
        with torch.no_grad(): model(M["cache"]["ssdc"][0][0][:2].to(DEVICE))
        h1.remove(); h2.remove()
        gate("FINAL readout equals model.norm's own output", key, torch.equal(got["a"], got["b"]))
    else:
        gate("FINAL readout uses fc_norm per token (pooled model)", key, fn_name == "fc_norm", fn_name)

    # first-layer numerics: must be RB-1b's cuDNN 9.10.2 values (the September numerics)
    if DEVICE == "cuda" and not TEST_MODE:
        v, nb = first_layer_rel_diff(M); exp_ = FIRST_LAYER_910[key]
        M["first_layer_rel_diff"] = v
        gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value (TF32-level kernel, as in September)", key,
             nb == BATCH_SIZE and abs(v / exp_ - 1) <= 0.01,
             f"{v:.4e} vs {exp_:.4e} (bit-identical: {v == exp_}; batch {nb})")
    else:
        gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value", key, True, "test mode / CPU", skip=True)

    # replay gates: cached path == the repo's streamed evaluate_ssdc
    def streamed_ssdc(ctx):
        torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
        with ctx:
            s, _ = evaluate_ssdc(model, processor, DS_SSDC, source, RPI=True, number_images=N_SSDC,
                                 batch_size=BATCH_SIZE, metric=METRIC, half=False, num_workers=0)
        return s
    for nm_, mk in [("baseline", NullCtx), ("Cref", lambda: RopeComposite(model, source, [(WINDOW, IDEN)]))]:
        a = streamed_ssdc(mk()); b = measure(M, mk())["ssdc"][:N_BLOCKS]
        d = max(abs(x - y) for x, y in zip(a, b))
        gate(f"cached replay reproduces streamed evaluate_ssdc [{nm_}]", key, d <= EXACT_TOL, f"max |delta| {d:.2e}")
    M["cache_digest"] = cache_digest(M["cache"]["ssdc"])
    return M

In [14]:
# bookkeeping: one file per model, saved after every block
RESULTS = {}
CFG_FP = {"n_images": N_SSDC, "batch": BATCH_SIZE, "seed": TORCH_SEED, "dataset": SSDC_DATASET, "metric": METRIC,
          "window": WINDOW, "post": POST, "p_ref": P_REF, "p_single": P_SINGLE, "noise_at": list(NOISE_AT),
          "probe": [PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC], "noise": [NOISE_SEEDS, NOISE_BOOT],
          "test_mode": TEST_MODE}
SESSION = {"session": RUN_STAMP, "torch": torch.__version__, "cudnn": ENV["cudnn_reported"], "python": sys.executable,
           "gpu": ENV["gpu"], "repo_commit": REPO_COMMIT}

def model_file(key): return OUT_DIR / f"rb2_{key}.json"

def load_R(key):
    f = model_file(key)
    if not (RESUME and f.exists()): return None
    R = json.loads(f.read_text())
    if R.get("config") != CFG_FP:
        raise SystemExit(f"{f.name} was written with a different configuration; move it away or set RB2_RESUME=0.\n"
                         f"  file: {R.get('config')}\n  now:  {CFG_FP}")
    return R

def save_R(key, R):
    R["saved_utc"] = datetime.now(timezone.utc).isoformat()
    tmp = model_file(key).with_suffix(".json.tmp")
    tmp.write_text(json.dumps(R, indent=1, default=float)); os.replace(tmp, model_file(key))

def single_name(w, k, h): return f"{w}_b{k}_h{h}"
def single_readouts(k): return P_SINGLE if k < N_BLOCKS - 1 else [FINAL]
def windows_done(R, nh):
    return [w for w in WINDOW_ORDER if all(single_name(w, k, h) in R["conditions"] for k in POST for h in range(nh))]

## Gates on the saved results
All of these read the saved file, so a resumed or re-loaded run still checks every condition. "Bit for bit" means
the values are identical, not just close.

In [15]:
def _maxdiff(a, b): return max((abs(x - y) for x, y in zip(a, b)), default=0.0)

def check_refs(key, R):
    """References vs W5 and the canonical RB-0. Returns True if every reference gate passed (or was skipped)."""
    c = R["conditions"]; ok = []
    A_, rb = ANCH.get(key), RB0.get(key) or {}
    exp_hooks = {"baseline": 0, "Cref": len(WINDOW), "Dref_row": len(WINDOW), "Dref_col": len(WINDOW)}
    ok.append(gate("references registered the expected hooks and saw every image", key,
                   all(c[n]["hooks"] == exp_hooks[n] and c[n]["n_images"] == N_SSDC for n in exp_hooks),
                   ", ".join(f"{n}:{c[n]['hooks']}h/{c[n]['n_images']}img" for n in exp_hooks)))
    if A_ and not TEST_MODE:
        d = max(_maxdiff(c[n]["ssdc"][:N_BLOCKS], A_["ssdc"][n]) for n in W5_REF)
        ok.append(gate("reference SSDC equals W5 bit for bit (baseline, Cref, Dref_row, Dref_col; 12 readouts each)", key,
                       d <= EXACT_TOL, f"max |delta| {d:.1e}"))
        pairs = [(c[n]["probes"][str(b)][ax], A_["probes"][n][b][ax]) for n in W5_REF for b in range(7, 12) for ax in ("row", "col")
                 if b in A_["probes"][n]]
        d = max((abs(x - y) for x, y in pairs), default=float("nan"))
        ok.append(gate("reference probes equal W5 bit for bit (4 references x blocks 7-11 x row/col)", key,
                       len(pairs) == 40 and d <= EXACT_TOL, f"max |delta| {d * 100:.2f} pts over {len(pairs)}"))
        npairs = []
        for n in ("Dref_row", "Dref_col"):
            for b, nz in A_["noise"][n].items():
                for ax in ("row", "col"):
                    mine = c[n]["probes"][str(b)].get("noise", {}).get(ax, {})
                    for f_ in ("boot_sd", "seed_sd", "combined_sd"):
                        npairs.append((mine.get(f_), nz.get(ax, {}).get(f_)))
        if npairs:
            ok.append(gate("probe noise equals W5 bit for bit where W5 measured it (Dref, block 11)", key,
                           all(a is not None and a == b for a, b in npairs), f"{len(npairs)} values"))
    elif not TEST_MODE:
        ok.append(gate("W5 anchors available", key, False, "no committed W5 file"))
    if rb.get("A"):
        ca = rb["A"]["conditions"]
        d = max(_maxdiff(c["Cref"]["ssdc"], ca["Cref"]["ssdc"]), _maxdiff(c["baseline"]["ssdc"], ca["baseline"]["ssdc"]))
        ok.append(gate("Cref and baseline SSDC equal RB-0 part A bit for bit (13 readouts)", key, d <= EXACT_TOL, f"max |delta| {d:.1e}"))
    if rb.get("B"):
        cb = rb["B"]["conditions"]; d, nv, noise_ok = 0.0, 0, True
        for n in ("baseline", "Dref_row", "Dref_col"):
            d = max(d, _maxdiff(c[n]["ssdc"], cb[n]["ssdc"]))
            for b in P_REF:
                for ax in ("row", "col"):
                    d = max(d, abs(c[n]["probes"][str(b)][ax] - cb[n]["probes"][str(b)][ax])); nv += 1
                if n != "baseline" and b in NOISE_AT:
                    noise_ok &= c[n]["probes"][str(b)].get("noise") == cb[n]["probes"][str(b)].get("noise")
        ok.append(gate("baseline and Dref SSDC + probes equal RB-0 part B bit for bit (blocks 7-11 and output of 11)", key,
                       d <= EXACT_TOL, f"max |delta| {d:.1e} over {nv} probes + 39 SSDC"))
        ok.append(gate("Dref probe noise equals RB-0 part B bit for bit (entering block 11 and output of block 11)", key, noise_ok))
    nz = [c["Cref"]["probes"][str(b)].get("noise", {}).get(ax, {}).get("combined_sd") for b in NOISE_AT for ax in ("row", "col")]
    ok.append(gate("Cref probe noise measured (new in RB-2) and finite", key,
                   all(v is not None and np.isfinite(v) for v in nz), "combined SD (pts): " + ", ".join(f"{v * 100:.2f}" for v in nz if v is not None)))
    return all(ok)

def check_singles(key, R):
    c = R["conditions"]; nh = R["n_heads"]; rb = RB0.get(key) or {}
    done = windows_done(R, nh)
    gate("windows with every single head measured", key, set(WINDOWS) <= set(done),
         f"done {done}; requested {WINDOWS}; {len([n for n in c if n[:4] in ('cref', 'row_', 'col_')])} single-head conditions")
    names = [(w, k, h) for w in done for k in POST for h in range(nh)]
    if not names: return
    bad_h = [single_name(*t) for t in names if c[single_name(*t)]["hooks"] != len(WINDOW) + 1 or c[single_name(*t)]["n_images"] != N_SSDC]
    gate("every single-head condition registered window + 1 hooks and saw every image", key, not bad_h, ",".join(bad_h[:5]))
    worst = max(_maxdiff(c[single_name(w, k, h)]["ssdc"][:k + 1], c[REFN[w]]["ssdc"][:k + 1]) for w, k, h in names)
    gate("single-head masks leave every readout at or before their block bit-identical to the window reference", key,
         worst <= EXACT_TOL, f"max |delta| {worst:.1e} over {len(names)} conditions")
    inert = [single_name(w, k, h) for w, k, h in names if c[single_name(w, k, h)]["ssdc"][FINAL] == c[REFN[w]]["ssdc"][FINAL]]
    gate("every single-head mask changes the SSDC at the output of block 11 (no inert mask)", key, not inert, ",".join(inert[:5]))
    rd_bad = [single_name(w, k, h) for w, k, h in names if sorted(c[single_name(w, k, h)]["probes"]) != sorted(str(r) for r in single_readouts(k))]
    gate("single heads probed at the planned readouts (blocks 6-10: entering 11 + output of 11; block 11: output of 11)", key,
         not rd_bad, ",".join(rd_bad[:5]))
    if "cref" in done and rb.get("A"):
        ca = rb["A"]["conditions"]
        pairs = [(c[single_name("cref", k, h)]["ssdc"], ca.get(f"s_b{k}_h{h}", {}).get("ssdc")) for k in POST for h in range(nh)]
        miss = sum(b is None for _, b in pairs)
        d = max((_maxdiff(a, b) for a, b in pairs if b is not None), default=float("nan"))
        gate("every cref-window single head's SSDC equals RB-0 part A bit for bit (13 readouts)", key,
             miss == 0 and d <= EXACT_TOL, f"max |delta| {d:.1e} over {len(pairs) - miss} heads; {miss} missing in RB-0")
    if rb.get("B"):
        cb = rb["B"]["conditions"]; d, n_s, n_p = 0.0, 0, 0
        for w in [x for x in ("row", "col") if x in done]:
            for k in POST:
                for h in range(nh):
                    o = cb.get(f"s_{w}_b{k}_h{h}")
                    if o is None: continue
                    m = c[single_name(w, k, h)]; n_s += 1
                    d = max(d, _maxdiff(m["ssdc"], o["ssdc"]))
                    for r in single_readouts(k):
                        for ax in ("row", "col"):
                            d = max(d, abs(m["probes"][str(r)][ax] - o["probes"][str(r)][ax])); n_p += 1
        if n_s:
            gate("row/col single heads that RB-0 also ran equal RB-0 part B bit for bit (SSDC + probes)", key,
                 d <= EXACT_TOL, f"max |delta| {d:.1e} over {n_s} conditions ({n_p} probes)")
    if not TEST_MODE:
        ses = R.get("sessions", [])
        gate("every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002", key,
             bool(ses) and all(s.get("cudnn") == EXPECT_CUDNN and s.get("torch") == EXPECT_TORCH for s in ses),
             f"{len(ses)} session(s): " + ", ".join(f"{s['session']}:{s.get('cudnn')}" for s in ses))

def attention_level_gate(M, window_name):
    """Masking one (block, head) after the window changes that head's attention output and no other."""
    key, model, source, nh = M["key"], M["model"], M["source"], M["n_heads"]
    A_ = ANCH.get(key); hb = (A_["stageC"][0] if (A_ and not TEST_MODE) else 0); kb = POST[0]
    tf = {"cref": IDEN, "row": M["ROWM"], "col": M["COLM"]}[window_name]
    grab = {}
    proj = get_block_attention(get_vit_blocks(model, source)[kb], source).proj
    hk = proj.register_forward_pre_hook(lambda m, args: grab.__setitem__("x", args[0].detach().clone()))
    xb = M["cache"]["ssdc"][0][0][:4].to(DEVICE)
    try:
        with torch.no_grad():
            with RopeComposite(model, source, [(WINDOW, tf)]): model(xb)
            ref_x = grab["x"]
            with win_plus(model, source, {kb: [hb]}, nh, window_tf=tf): model(xb)
            new_x = grab["x"]
    finally:
        hk.remove()
    hd = ref_x.shape[-1] // nh
    same = all(torch.equal(ref_x[..., h * hd:(h + 1) * hd], new_x[..., h * hd:(h + 1) * hd]) for h in range(nh) if h != hb)
    changed = not torch.equal(ref_x[..., hb * hd:(hb + 1) * hd], new_x[..., hb * hd:(hb + 1) * hd])
    gate(f"[{window_name} window] masking (block {kb}, head {hb}) changes that head's attention output and leaves the "
         f"other {nh - 1} bit-identical", key, same and changed)

## Measurement
Per model: the four references (baseline, Cref, Dref_row, Dref_col; probes at blocks 7-11 and the output of
block 11; probe noise on the three window references), the reference gates, then every single head under each
window. If a reference gate fails, the model is not continued (`RB2_STRICT=1`), so a broken setting cannot cost an
hour. Saves after every block.

In [16]:
def run_model(M):
    key, model, source, nh = M["key"], M["model"], M["source"], M["n_heads"]
    H, W = M["grid"]
    R = load_R(key) or {"model": key, "name": MODEL_SPECS[key]["name"], "notebook": "RB-2", "config": CFG_FP,
                        "n_heads": nh, "grid": [H, W], "row_planes": M["row_planes"], "col_planes": M["col_planes"],
                        "readouts": "0-11 = input of block b after its first LayerNorm (as W2-RB-0); 12 = output of block 11 "
                                    f"through model.{M['final_norm_name']}", "conditions": {}, "sessions": []}
    if R["conditions"]:
        print(f"  resuming {model_file(key).name}: {len(R['conditions'])} conditions already saved")
    R["sessions"].append({**SESSION, "first_layer_rel_diff": M.get("first_layer_rel_diff"), "cache_digest": M.get("cache_digest")})
    TF = {"cref": IDEN, "row": M["ROWM"], "col": M["COLM"]}
    t0 = time.time(); n_new = [0]

    def run(name, ctx, readouts, noise_at=(), extra=None):
        if name in R["conditions"]: return R["conditions"][name]
        r = measure(M, ctx, capture=readouts)
        probes = {str(b): probe_block(r["acts"][b], H, W, b, heads=("row", "col"), noise=(b in noise_at)) for b in readouts}
        del r["acts"]
        rec = {"ssdc": r["ssdc"], "probes": probes, "hooks": r["hooks"], "n_images": r["n_images"],
               "forward_s": r["runtime_s"], "session": RUN_STAMP}
        if extra: rec.update(extra)
        R["conditions"][name] = rec; n_new[0] += 1
        return rec

    print(f"\n--- references: {key} ---", flush=True)
    for name, ctx, nz in [("baseline", NullCtx(), ()),
                          ("Cref", RopeComposite(model, source, [(WINDOW, IDEN)]), NOISE_AT),
                          ("Dref_row", RopeComposite(model, source, [(WINDOW, M["ROWM"])]), NOISE_AT),
                          ("Dref_col", RopeComposite(model, source, [(WINDOW, M["COLM"])]), NOISE_AT)]:
        rec = run(name, ctx, P_REF, noise_at=nz)
        p = rec["probes"]
        print(f"    {name:9s} b11 row {p['11']['row'] * 100:5.1f} col {p['11']['col'] * 100:5.1f} | out11 row "
              f"{p['12']['row'] * 100:5.1f} col {p['12']['col'] * 100:5.1f} | SSDC b11 {rec['ssdc'][11]:.6f}  {time.time() - t0:.0f}s", flush=True)
    save_R(key, R)
    refs_ok = check_refs(key, R)
    for w in ("cref", "row"):
        attention_level_gate(M, w)
    if not refs_ok and STRICT and not TEST_MODE:
        gate("single heads measured", key, False, "STOPPED: a reference gate failed (RB2_STRICT=1). Send the gate report.")
        return R

    n_todo = sum(single_name(w, k, h) not in R["conditions"] for w in WINDOWS for k in POST for h in range(nh))
    print(f"\n--- single heads: {key}, windows {WINDOWS}, {n_todo} conditions to measure ---", flush=True)
    t1 = time.time(); n_at_start = n_new[0]
    for w in WINDOWS:
        ref = R["conditions"][REFN[w]]
        for k in POST:
            if all(single_name(w, k, h) in R["conditions"] for h in range(nh)): continue
            for h in range(nh):
                run(single_name(w, k, h), win_plus(model, source, {k: [h]}, nh, window_tf=TF[w]), single_readouts(k),
                    extra={"window": w, "block": k, "head": h})
            save_R(key, R)
            r_ = str(single_readouts(k)[0])
            costs = [((ref["probes"][r_]["row"] - R["conditions"][single_name(w, k, h)]["probes"][r_]["row"]) * 100,
                      (ref["probes"][r_]["col"] - R["conditions"][single_name(w, k, h)]["probes"][r_]["col"]) * 100, h) for h in range(nh)]
            br, bc = max(costs), max(costs, key=lambda t: t[1])
            done_n = n_new[0] - n_at_start; el = time.time() - t1
            eta = el / max(done_n, 1) * max(n_todo - done_n, 0)
            print(f"    [{w}] block {k:2d} done | {'b11' if k < 11 else 'out11'} largest row cost h{br[2]} {br[0]:5.2f} pts, "
                  f"largest col cost h{bc[2]} {bc[1]:5.2f} pts | {done_n}/{n_todo} | {el / 60:.1f} min, ~{eta / 60:.0f} min left", flush=True)
    R["complete"] = set(windows_done(R, nh)) == set(WINDOW_ORDER)
    R["runtime_s_this_session"] = round(time.time() - t0, 1)
    save_R(key, R)
    check_singles(key, R)
    gate("model finished", key, True, f"{n_new[0]} new conditions this session, {R['runtime_s_this_session'] / 60:.1f} min; "
                                       f"all three windows complete: {R['complete']}")
    return R

In [17]:
T_RUN = time.time()
for key in MODELS_RUN:
    R0 = load_R(key)
    if R0 and set(WINDOWS) <= set(windows_done(R0, R0["n_heads"])) and all(n in R0["conditions"] for n in REFN.values()):
        print(f"\n{key}: every requested window already measured in {model_file(key).name}; loading and re-checking")
        RESULTS[key] = R0; check_refs(key, R0); check_singles(key, R0)
        continue
    try:
        M = setup_model(key)
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", key, False, repr(ex)[:200]); continue
    setup_fail = [g for g in GATES if g["scope"] == key and g["status"] == "FAIL"]
    if setup_fail and STRICT and not TEST_MODE:
        gate("model measured", key, False, f"STOPPED: {len(setup_fail)} setup gate(s) failed (RB2_STRICT=1): "
                                           + "; ".join(g["gate"] for g in setup_fail)[:300])
        del M; continue
    try:
        RESULTS[key] = run_model(M)
    except Exception as ex:
        traceback.print_exc(); gate("model measured", key, False, repr(ex)[:200])
    del M
    if DEVICE == "cuda": torch.cuda.empty_cache()
print(f"\nmeasurement finished in {(time.time() - T_RUN) / 60:.1f} min")


naver  (vit_base_patch16_rope_224.naver_in1k)


    cached 1000 images in 5s
  [PASS] naver: forward grid is 14x14: 14x14


  heads=12 prefix=1 rotate_half=False final readout via model.norm
  [PASS] naver: num_heads matches W5: 12 vs 12


  [PASS] naver: num_heads matches RB-0: 12 vs 12


  [PASS] naver: rope rows equal patch tokens: 196 vs 196


  [PASS] naver: rotary table has no head dimension (every head in a block uses the same frequencies): shape (196, 128); 32 planes per head


... (213 outputs left out)

  [PASS] reg1_gap: row/col single heads that RB-0 also ran equal RB-0 part B bit for bit (SSDC + probes): max |delta| 0.0e+00 over 32 conditions (120 probes)


  [PASS] reg1_gap: every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002: 1 session(s): 2026-11-05T04-31-07Z:91002


  [PASS] reg1_gap: model finished: 220 new conditions this session, 52.5 min; all three windows complete: True



measurement finished in 184.2 min


## Analysis
Reads only the saved files (re-runnable on its own after a restart). Costs are probe points lost against the
window's reference: `cref` against Cref, `row` against Dref_row, `col` against Dref_col (Figure 4d / Table 14's
definition). SSDC costs are against Cref. Top-1 costs are RB-0 part C (first 5,000 ILSVRC images, against the
0-5 window), with paired per-image SEs.

In [18]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

def perm_spearman(x, y, n_perm=PERM_N, seed=0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    rx, ry = rankdata(x), rankdata(y)
    if np.std(rx) == 0 or np.std(ry) == 0: return float("nan"), float("nan")   # a constant column: undefined
    rho = float(np.corrcoef(rx, ry)[0, 1])
    g = np.random.default_rng(seed)
    null = np.array([np.corrcoef(rx, g.permutation(ry))[0, 1] for _ in range(n_perm)])
    return rho, float((np.sum(np.abs(null) >= abs(rho)) + 1) / (n_perm + 1))

def spearman(x, y):
    rx, ry = rankdata(x), rankdata(y)
    if np.std(rx) == 0 or np.std(ry) == 0: return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])

def paired(a, b):
    d = a.astype(float) - b.astype(float); return float(d.mean()), float(d.std(ddof=1) / np.sqrt(len(d)))

def label(rc, cc, bar):
    if rc > bar and rc - cc > bar: return "row"
    if cc > bar and cc - rc > bar: return "col"
    if rc > bar or cc > bar: return "mixed"
    return "none"

def tag(r): return "b11" if r == 11 else "out11"

ANALYSIS, TABLES = {}, {k: [] for k in ("axis_map", "noise", "index_groups", "scores_vs_top1", "top_two")}
for key in MODELS_RUN:
    R = RESULTS.get(key) or (json.loads(model_file(key).read_text()) if model_file(key).exists() else None)
    if not R or not all(n in R["conditions"] for n in REFN.values()):
        print(f"{key}: no references saved, skipped"); continue
    c, nh = R["conditions"], R["n_heads"]; done = windows_done(R, nh); out = {"windows_done": done}
    rb = RB0.get(key) or {}
    def pr(name, r, ax): return c[name]["probes"][str(r)][ax]
    def sd(name, r, ax): return c[name]["probes"][str(r)].get("noise", {}).get(ax, {}).get("combined_sd")
    # noise bars
    bar = {}
    for r in NOISE_AT:
        bar[("axis", r)] = 300 * max(sd("Dref_row", r, "row"), sd("Dref_col", r, "col"))
        bar[("cref", r)] = 300 * max(sd("Cref", r, "row"), sd("Cref", r, "col"))
        for n in REFN.values():
            for ax in ("row", "col"):
                TABLES["noise"].append({"model": key, "reference": n, "readout": tag(r), "probe": ax,
                                        "accuracy_pts": pr(n, r, ax) * 100, "combined_sd_pts": sd(n, r, ax) * 100,
                                        "bar_3sd_pts": 300 * sd(n, r, ax)})
    out["bars_pts"] = {f"{w}_{tag(r)}": v for (w, r), v in bar.items()}
    # top-1 per single head from RB-0 part C
    top1 = {}
    if rb.get("C") and rb.get("P") is not None:
        cc_, Pm = rb["C"]["conditions"], rb["P"]; win = cc_["window"]["acc"]
        for k in POST:
            for h in range(nh):
                s = f"s_b{k}_h{h}"
                if s in cc_ and s in Pm:
                    m_, se_ = paired(Pm["window"], Pm[s])
                    top1[(k, h)] = ((win - cc_[s]["acc"]) * 100, se_ * 100)
    # the map
    rows = []
    for k in POST:
        for h in range(nh):
            row = {"model": key, "block": k, "head": h, "primary": tag(single_readouts(k)[0])}
            for r in single_readouts(k):
                t_ = tag(r)
                if "cref" in done:
                    s = single_name("cref", k, h)
                    a, b = (pr("Cref", r, "row") - pr(s, r, "row")) * 100, (pr("Cref", r, "col") - pr(s, r, "col")) * 100
                    row.update({f"cref_row_cost_{t_}": a, f"cref_col_cost_{t_}": b, f"probe_score_{t_}": a + b,
                                f"cref_favours_{t_}": label(a, b, bar[("cref", r)])})
                if "row" in done and "col" in done:
                    sr, sc = single_name("row", k, h), single_name("col", k, h)
                    rc = (pr("Dref_row", r, "row") - pr(sr, r, "row")) * 100
                    cc = (pr("Dref_col", r, "col") - pr(sc, r, "col")) * 100
                    row.update({f"row_cost_{t_}": rc, f"col_cost_{t_}": cc, f"axis_score_{t_}": rc + cc,
                                f"rowwin_col_cost_{t_}": (pr("Dref_row", r, "col") - pr(sr, r, "col")) * 100,
                                f"colwin_row_cost_{t_}": (pr("Dref_col", r, "row") - pr(sc, r, "row")) * 100,
                                f"favours_{t_}": label(rc, cc, bar[("axis", r)])})
            src = c.get(single_name("cref", k, h)) or (rb.get("A") or {}).get("conditions", {}).get(f"s_b{k}_h{h}")
            if src:
                row["ssdc_cost_b11"] = c["Cref"]["ssdc"][11] - src["ssdc"][11]
                row["ssdc_cost_out11"] = c["Cref"]["ssdc"][FINAL] - src["ssdc"][FINAL]
            if (k, h) in top1:
                row["top1_cost"], row["top1_se"] = top1[(k, h)]
            p_ = row["primary"]
            for f_ in ("row_cost", "col_cost", "favours", "probe_score", "cref_favours"):
                if f"{f_}_{p_}" in row: row[f_] = row[f"{f_}_{p_}"]
            rows.append(row)
    df = pd.DataFrame(rows); TABLES["axis_map"] += rows
    # specialists
    if "favours" in df:
        cnt = {}
        for grp, sub in (("blocks 6-10 (b11)", df[df.block < 11]), ("block 11 (out11)", df[df.block == 11])):
            cnt[grp] = {l: int((sub.favours == l).sum()) for l in ("row", "col", "mixed", "none")}
        cnt["all heads at out11"] = {l: int((df.favours_out11 == l).sum()) for l in ("row", "col", "mixed", "none")}
        out["favours_counts"] = cnt
        spec = df[df.favours.isin(["row", "col", "mixed"])].copy()
        spec["max_cost"] = spec[["row_cost", "col_cost"]].max(axis=1)
        out["specialists"] = spec.sort_values("max_cost", ascending=False)[
            ["block", "head", "primary", "row_cost", "col_cost", "favours"] + (["top1_cost"] if "top1_cost" in spec else [])
        ].round(3).to_dict("records")
    # Table 14 index groups: how much comes from one block
    if "row_cost_b11" in df:
        A_ = ANCH.get(key); gr = {}
        heads14 = sorted(A_["D"]) if (A_ and A_["D"]) else []
        src_name = "W5 stage D"
        if not heads14 and rb.get("B"):
            heads14 = sorted({int(n.split("_h")[1]) for n in rb["B"]["conditions"] if n.startswith("grp_row_h")}); src_name = "RB-0 part B"
        for h in heads14:
            if A_ and h in A_["D"]:
                Dg, Dr = A_["D"][h], A_["probes"]
                g_row = {b: (Dr["Dref_row"][b]["row"] - Dg["row"][b]["row"]) * 100 for b in range(7, 12)}
                g_col = {b: (Dr["Dref_col"][b]["col"] - Dg["col"][b]["col"]) * 100 for b in range(7, 12)}
            else:
                cb = rb["B"]["conditions"]
                g_row = {b: (cb["Dref_row"]["probes"][str(b)]["row"] - cb[f"grp_row_h{h}"]["probes"][str(b)]["row"]) * 100 for b in range(7, 12)}
                g_col = {b: (cb["Dref_col"]["probes"][str(b)]["col"] - cb[f"grp_col_h{h}"]["probes"][str(b)]["col"]) * 100 for b in range(7, 12)}
            sub = df[(df["head"] == h) & (df.block < 11)].set_index("block")
            b11h = df[(df["head"] == h) & (df.block == 11)].iloc[0]
            rec = {"model": key, "head": h, "source": src_name,
                   "table14_row": float(np.mean(list(g_row.values()))), "table14_col": float(np.mean(list(g_col.values())))}
            for ax, g_ in (("row", g_row), ("col", g_col)):
                cn = f"{ax}_cost_b11"; best_k = int(sub[cn].idxmax()); best = float(sub.loc[best_k, cn])
                rec.update({f"{ax}_group_b11": g_[11], f"{ax}_sum_singles_6_10": float(sub[cn].sum()),
                            f"{ax}_best_block": best_k, f"{ax}_best_cost": best,
                            f"{ax}_best_share": best / g_[11] if g_[11] > bar[("axis", 11)] else float("nan"),
                            f"{ax}_best_favours": sub.loc[best_k, "favours_b11"]})
            rec["per_block_b11"] = {int(kk): [round(float(sub.loc[kk, "row_cost_b11"]), 2), round(float(sub.loc[kk, "col_cost_b11"]), 2),
                                              sub.loc[kk, "favours_b11"]] for kk in sub.index}
            rec["block11_head_out11"] = [round(float(b11h.row_cost_out11), 2), round(float(b11h.col_cost_out11), 2), b11h.favours_out11]
            gr[str(h)] = rec; TABLES["index_groups"].append({k_: v for k_, v in rec.items() if not isinstance(v, (dict, list))})
        out["index_groups"] = gr
    # which score predicts a head's top-1 cost? (pre-registered rule above)
    d6 = df[(df.block < 11)].dropna(subset=["top1_cost"]) if "top1_cost" in df else df.iloc[0:0]
    if len(d6) and "probe_score_b11" in d6:
        res = {"n_heads": int(len(d6))}
        for sname, col in (("ssdc_b11", "ssdc_cost_b11"), ("probe_b11", "probe_score_b11"), ("axis_b11", "axis_score_b11")):
            if col in d6: res[sname] = perm_spearman(d6[col], d6["top1_cost"])
        g = np.random.default_rng(DELTA_SEED); x1, x2, y = d6["probe_score_b11"].values, d6["ssdc_cost_b11"].values, d6["top1_cost"].values
        deltas = []
        for _ in range(DELTA_BOOT):
            i = g.integers(0, len(y), len(y)); deltas.append(spearman(x1[i], y[i]) - spearman(x2[i], y[i]))
        deltas = np.array([v for v in deltas if np.isfinite(v)])
        lo, hi = (float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))) if len(deltas) >= 0.9 * DELTA_BOOT else (float("nan"), float("nan"))
        res["delta_probe_minus_ssdc"] = [res["probe_b11"][0] - res["ssdc_b11"][0], lo, hi]
        res["delta_boot_valid"] = int(len(deltas))      # resamples where both correlations were defined
        d11 = df[(df.block == 11)].dropna(subset=["top1_cost"])
        if len(d11) > 2:
            res["block11_out11"] = {"ssdc": spearman(d11["ssdc_cost_out11"], d11["top1_cost"]),
                                    "probe": spearman(d11["probe_score_out11"], d11["top1_cost"]) if "probe_score_out11" in d11 else None,
                                    "n": int(len(d11))}
        dall = df.dropna(subset=["top1_cost"])
        kc = dall.loc[dall.top1_cost.idxmax()]
        res["costliest_head_top1"] = [int(kc.block), int(kc["head"]), float(kc.top1_cost), float(kc.top1_se)]
        for sname, col in (("ssdc_b11", "ssdc_cost_b11"), ("probe_b11", "probe_score_b11")):
            res[f"top5_by_{sname}"] = [(int(a), int(b)) for a, b in d6.sort_values(col, ascending=False)[["block", "head"]].values[:5]]
        out["scores_vs_top1"] = res
        TABLES["scores_vs_top1"].append({"model": key, "n": res["n_heads"],
                                         **{f"rho_{s}": res[s][0] for s in ("ssdc_b11", "probe_b11", "axis_b11") if s in res},
                                         **{f"p_{s}": res[s][1] for s in ("ssdc_b11", "probe_b11", "axis_b11") if s in res},
                                         "delta": res["delta_probe_minus_ssdc"][0], "delta_lo": res["delta_probe_minus_ssdc"][1],
                                         "delta_hi": res["delta_probe_minus_ssdc"][2]})
    # do the probes separate the two top heads by SSDC (blocks 6-10)? ViT-S: (7,2) vs (7,4)
    if "ssdc_cost_b11" in df:
        t2 = df[df.block < 11].sort_values("ssdc_cost_b11", ascending=False).head(2)
        recs = []
        for _, r_ in t2.iterrows():
            recs.append({"model": key, "block": int(r_.block), "head": int(r_["head"]), "ssdc_cost_b11": float(r_.ssdc_cost_b11),
                         **{f: (float(r_[f]) if isinstance(r_.get(f), (int, float, np.floating)) else r_.get(f))
                            for f in ("probe_score_b11", "cref_row_cost_b11", "cref_col_cost_b11", "row_cost_b11", "col_cost_b11",
                                      "favours_b11", "top1_cost", "top1_se") if f in r_}})
        if len(recs) == 2 and rb.get("P") is not None:
            a_, b_ = f"s_b{recs[0]['block']}_h{recs[0]['head']}", f"s_b{recs[1]['block']}_h{recs[1]['head']}"
            if a_ in rb["P"] and b_ in rb["P"]:
                m_, se_ = paired(rb["P"][b_], rb["P"][a_])     # cost(first) - cost(second)
                recs[0]["top1_cost_minus_second"] = [m_ * 100, se_ * 100]
        out["top_two_by_ssdc"] = recs; TABLES["top_two"] += recs
    ANALYSIS[key] = out

# pre-registered decision for later runs
_cls = [k for k in ("naver", "small", "reg1_gap") if k in ANALYSIS and "scores_vs_top1" in ANALYSIS[k]]
_above = [k for k in _cls if ANALYSIS[k]["scores_vs_top1"]["delta_probe_minus_ssdc"][1] > 0]
_undef = [k for k in _cls if not np.isfinite(ANALYSIS[k]["scores_vs_top1"]["delta_probe_minus_ssdc"][1])]
DECISION = {"rule": RULE_NEXT, "models_evaluated": _cls, "models_where_probe_beats_ssdc": _above,
            "ranking_for_later_runs": ("undecided: needs naver, small and reg1_gap" if len(_cls) < 3 else
                                       f"undecided: bootstrap interval undefined for {_undef}" if _undef else
                                       "probe_b11" if len(_above) >= 2 else "ssdc_b11")}

In [19]:
def show(title, obj):
    print("\n" + title); print("-" * len(title))
    if isinstance(obj, pd.DataFrame): print(obj.to_string(index=False) if len(obj) else "(empty)")
    else: print(json.dumps(obj, indent=1, default=float))

for key, out in ANALYSIS.items():
    print("\n" + "=" * 78 + f"\n{key}  ({MODEL_SPECS[key]['label']}) windows done: {out['windows_done']}\n" + "=" * 78)
    print("noise bars (3 SD, points):", {k: round(v, 2) for k, v in out["bars_pts"].items()})
    if "favours_counts" in out:
        print("single heads by favoured axis (primary readout):", out["favours_counts"])
        show("specialists (row / col / mixed), largest cost first", pd.DataFrame(out["specialists"]).head(15))
    if "index_groups" in out:
        gt = pd.DataFrame([{k: v for k, v in g.items() if not isinstance(v, (dict, list))} for g in out["index_groups"].values()])
        show("Table 14 index groups vs their single heads (points; table14_* = the paper's 7-11 mean, the rest entering block 11)",
             gt.drop(columns=["model", "source"]).round(2))
        for h, g in out["index_groups"].items():
            print(f"   h{h}: per block [row cost, col cost, favours] {g['per_block_b11']} | block-11 head at out11 {g['block11_head_out11']}")
    if "scores_vs_top1" in out:
        s = out["scores_vs_top1"]
        print(f"\nwhich score predicts a head's top-1 cost? (blocks 6-10, n={s['n_heads']}; Spearman, permutation p)")
        for n in ("ssdc_b11", "probe_b11", "axis_b11"):
            if n in s: print(f"   {n:10s} rho {s[n][0]:+.3f}  p {s[n][1]:.4f}")
        d = s["delta_probe_minus_ssdc"]
        print(f"   probe - ssdc: {d[0]:+.3f}  [95% CI {d[1]:+.3f}, {d[2]:+.3f}]  -> {'probe better' if d[1] > 0 else 'not clearly better'}")
        if "block11_out11" in s: print("   block 11 (out11):", s["block11_out11"])
        print("   costliest head in top-1 (block, head, cost, SE):", s["costliest_head_top1"])
        print("   top 5 by SSDC:", s["top5_by_ssdc_b11"], "| top 5 by probe:", s["top5_by_probe_b11"])
    if "top_two_by_ssdc" in out:
        show("the two top heads by SSDC (blocks 6-10)", pd.DataFrame(out["top_two_by_ssdc"]).drop(columns=["model"]))
print("\nPRE-REGISTERED DECISION FOR LATER RUNS:", json.dumps(DECISION, indent=1))


naver  (naver ViT-B) windows done: ['cref', 'row', 'col']
noise bars (3 SD, points): {'axis_b11': 0.79, 'cref_b11': 0.75, 'axis_out11': 0.8, 'cref_out11': 0.77}
single heads by favoured axis (primary readout): {'blocks 6-10 (b11)': {'row': 4, 'col': 1, 'mixed': 0, 'none': 55}, 'block 11 (out11)': {'row': 0, 'col': 0, 'mixed': 0, 'none': 12}, 'all heads at out11': {'row': 4, 'col': 0, 'mixed': 0, 'none': 68}}

specialists (row / col / mixed), largest cost first
---------------------------------------------------
 block  head primary  row_cost  col_cost favours  top1_cost
     6     6     b11     2.982    -0.054     row       3.04
     6     1     b11     1.921    -0.023     row       2.92
     7     2     b11     1.559    -0.033     row       0.42
     6     4     b11    -0.048     0.972     col       2.14
     6     3     b11     0.811    -0.054     row      -0.90

Table 14 index groups vs their single heads (points; table14_* = the paper's 7-11 mean, the rest entering block 11)
-----

In [20]:
# figure: Figure 4d at single-head level (row cost vs column cost, primary readout)
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    keys = [k for k in ANALYSIS if "favours_counts" in ANALYSIS[k]]
    if keys:
        dfa = pd.DataFrame(TABLES["axis_map"])
        fig, axs = plt.subplots(1, len(keys), figsize=(4.2 * len(keys), 4.2), squeeze=False)
        colours = {"row": "#2f6db5", "col": "#d9822b", "mixed": "#7b4fa0", "none": "#a8a8a8"}
        for ax_, key in zip(axs[0], keys):
            d = dfa[dfa.model == key]; b = ANALYSIS[key]["bars_pts"]["axis_b11"]
            ax_.add_patch(plt.Rectangle((-b, -b), 2 * b, 2 * b, color="#e6e6e6", zorder=0))
            for lab_, sub in d.groupby("favours"):
                ax_.scatter(sub.row_cost, sub.col_cost, s=np.where(sub.block == 11, 18, 30), c=colours[lab_],
                            marker="o", label=lab_, edgecolors="none", zorder=2)
            top = d[d.block < 11].sort_values("ssdc_cost_b11", ascending=False).head(3)
            for _, r_ in top.iterrows():
                ax_.annotate(f"({int(r_.block)},{int(r_['head'])})", (r_.row_cost, r_.col_cost), fontsize=8,
                             xytext=(3, 3), textcoords="offset points")
            ax_.axhline(0, c="#888", lw=0.5); ax_.axvline(0, c="#888", lw=0.5)
            ax_.set_title(MODEL_SPECS[key]["label"], fontsize=10); ax_.set_xlabel("row cost (pts)"); ax_.set_ylabel("column cost (pts)")
            ax_.legend(fontsize=7, frameon=False)
        fig.suptitle("RB-2: every single (block, head) in blocks 6-11 (entering block 11; block-11 heads at the output of block 11). "
                     "Grey: 3-SD probe noise. Labels: top 3 by SSDC.", fontsize=8)
        fig.tight_layout(); fig.savefig(OUT_DIR / "rb2_axis_map.png", dpi=150); plt.close(fig)
        print("figure written: rb2_axis_map.png")
except Exception as ex:
    print("figure skipped:", repr(ex))

figure written: rb2_axis_map.png


## Gate report and export

In [21]:
_all = []
for f in sorted(OUT_DIR.glob("rb2_gates_session_*.json")):
    _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g      # a gate re-run in a later session: the latest result counts
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT  ({len(set(g['session'] for g in _all))} session(s), {len(_all)} gate calls, latest result per gate shown)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
n_skip = int((gdf.status == "SKIP").sum()) if len(gdf) else 0
complete = {k: (ANALYSIS.get(k) or {}).get("windows_done") == WINDOW_ORDER for k in MODELS_RUN}
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {n_skip} skipped")
print("models with all three windows measured:", complete)
print("=" * 78)
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: report them before anyone uses the numbers")
for _, r in gdf[gdf.status == "FAIL"].iterrows():
    print(f"   - [{r['scope']}] {r['gate']}: {r['detail']}")
print("=" * 78)

export = {
    "notebook": "RB-2", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "environment": ENV, "repo": str(repo),
    "repo_commit": REPO_COMMIT, "rb0_dir": str(_rb0) if _rb0 else None,
    "config": {**CFG_FP, "models": MODELS_RUN, "windows": WINDOWS, "strict": STRICT},
    "rules": {"bar": RULE_BAR, "label": RULE_LABEL, "readout": RULE_READOUT, "scores": RULE_SCORES, "next": RULE_NEXT},
    "decision": DECISION, "anchors_used": {"W5": {k: v["file"] for k, v in ANCH.items()}, "RB0_sha256": RB0_SHA256},
    "analysis": ANALYSIS, "gates": gdf.to_dict("records"),
    "gate_summary": {"total": len(gdf), "fail": n_fail, "skip": n_skip, "sessions": sorted(set(g["session"] for g in _all)),
                     "complete": complete},
}
(OUT_DIR / f"rb2_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=float))
gdf.to_csv(OUT_DIR / "rb2_gate_report.csv", index=False)
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"rb2_{name}.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("written:")
for p in sorted(OUT_DIR.iterdir()): print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")

GATE REPORT  (1 session(s), 168 gate calls, latest result per gate shown)

                                                                                                              gate       scope status                                                                                                                                                                                                     detail              session
                                                                                                    CUDA available         env   PASS                                                                                                                                                                                                            2026-11-05T04-31-07Z
                                                                                             torch is 2.10.0+cu128         env   PASS                                                                            